In [6]:
%pip install catboost

In [7]:
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import train_test_split
import pandas as pd

In [8]:
publication_dataframe = pd.read_csv('/content/test_set.csv')
publication_dataframe

,id,link,source,description,state,date_posted,date_crawler,price,rooms,area_sqm,...,floor,total_floors,has_elevator,has_balcony,has_parking,pets_allowed,deposit_amount,property_type,is_offer,currency
0,124,https://reality.bazos.sk/inzerat/193597775/na-...,bazos,"Title: Na prenájom zariadený 2,5 izbový byt ba...",fully_processed,2026-07-13,2026-08-07,1000.0,2.5,55.00,...,NaN,NaN,NaN,True,True,NaN,2000.0,entire_apartment,True,EUR
1,306,https://reality.bazos.sk/inzerat/193928275/na-...,bazos,Title: NA PRENÁJOM: Kompletne zariadený 2-izbo...,fully_processed,2026-07-23,2026-08-07,950.0,2.0,53.80,...,4.0,7.0,True,True,True,False,950.0,entire_apartment,True,EUR
2,27,https://reality.bazos.sk/inzerat/193356378/his...,bazos,Title: Historické bývanie v zámku Schloss Walt...,fully_processed,2026-07-11,2026-07-11,551.0,1.0,24.68,...,NaN,NaN,NaN,True,True,True,551.0,room_only,True,EUR
3,435,https://reality.bazos.sk/inzerat/194103479/na-...,bazos,"Title: Na prenájom krásny, tichý a slnečný 2-i...",fully_processed,2026-07-29,2026-08-07,1200.0,2.0,47.00,...,7.0,7.0,NaN,True,NaN,NaN,NaN,entire_apartment,True,EUR
4,163,https://reality.bazos.sk/inzerat/193664958/pre...,bazos,"Title: Prenájom 2 izbového bytu 54 m2, River P...",fully_processed,2026-07-15,2026-08-07,1400.0,2.0,54.00,...,7.0,9.0,True,True,True,NaN,NaN,entire_apartment,True,EUR
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
817,516,https://reality.bazos.sk/inzerat/194181523/pre...,bazos,"Title: Prenájom 2izbový byt L.Dérera, Bratisla...",fully_processed,2026-08-01,2026-08-07,890.0,2.0,58.00,...,5.0,11.0,True,True,NaN,NaN,1780.0,entire_apartment,True,EUR
818,291,https://reality.bazos.sk/inzerat/193908843/pre...,bazos,Title: Prenajom 3izb. bytu v BA projekte Ovocn...,fully_processed,2026-07-23,2026-08-07,1585.0,3.0,74.00,...,1.0,NaN,NaN,True,True,NaN,2700.0,entire_apartment,True,EUR
819,348,https://reality.bazos.sk/inzerat/194019404/241...,bazos,Title: 2418 SVETLÝ DVOJIZBOVÝ BYT S BALKÓNOM A...,fully_processed,2026-07-27,2026-08-07,1158.0,2.0,67.00,...,NaN,NaN,NaN,True,True,False,1000.0,entire_apartment,True,EUR
820,262,https://reality.bazos.sk/inzerat/193859658/2-i...,bazos,"Title: 2-IZBOVÝ BYT, RUŽINOV\nPrenájom priestr...",fully_processed,2026-07-21,2026-08-07,950.0,2.0,NaN,...,NaN,NaN,NaN,NaN,True,NaN,1750.0,entire_apartment,True,EUR


In [20]:
# Temporary line to test model's performance
publication_dataframe = publication_dataframe[(publication_dataframe["price"] < 2000) & (publication_dataframe["property_type"] != "room_only")]

In [11]:
text_columns = ['description']
cat_columns = ['building',
               'street',
               'district',
               'city',
               'country',
               'building_type',
               'building_name',
               'nearest_shopping_mall_name',
               'nearest_supermarket_name',
               'nearest_transport_stop_name',
               'property_type',
               'currency']

In [12]:
publication_dataframe.dropna(subset=['price'], inplace=True)
publication_dataframe.drop(columns=['id', 'link', 'source', 'state', 'date_posted', 'date_crawler'], inplace=True)
publication_dataframe[cat_columns] = publication_dataframe[cat_columns].fillna('NaN')

/tmp/ipykernel_2535/496573729.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  publication_dataframe.dropna(subset=['price'], inplace=True)
/tmp/ipykernel_2535/496573729.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  publication_dataframe.drop(columns=['id', 'link', 'source', 'state', 'date_posted', 'date_crawler'], inplace=True)
/tmp/ipykernel_2535/496573729.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexin

In [13]:
X, y = publication_dataframe.drop(columns=['price']), publication_dataframe['price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [14]:
train_pool = Pool(
    data=X_train,
    label=y_train,
    text_features=text_columns,
    cat_features=cat_columns
)

test_pool = Pool(
    data=X_test,
    label=y_test,
    text_features=text_columns,
    cat_features=cat_columns
)

In [15]:
model = CatBoostRegressor(
    iterations=2000,
    learning_rate=0.05,
    depth=6,
    eval_metric='RMSE',
    random_seed=42
)


In [16]:
model.fit(
    train_pool,
    eval_set=test_pool,
    early_stopping_rounds=50,
    verbose=100
)

0:	learn: 290.9739371	test: 287.7896757	best: 287.7896757 (0)	total: 500ms	remaining: 16m 39s
100:	learn: 117.6026553	test: 162.2285514	best: 162.2285514 (100)	total: 27.7s	remaining: 8m 40s
200:	learn: 81.4726668	test: 152.4858391	best: 152.4858391 (200)	total: 55.5s	remaining: 8m 16s
300:	learn: 61.9818124	test: 150.8356800	best: 150.6203400 (293)	total: 1m 22s	remaining: 7m 47s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 150.1947931
bestIteration = 340

Shrink model to first 341 iterations.


CatBoostRegressor(depth=6, eval_metric='RMSE', iterations=2000, learning_rate=0.05, loss_function='RMSE', random_seed=42)

In [17]:
model.get_feature_importance()

array([6.17029190e+01, 1.99444651e+01, 1.00491832e+01, 2.49528621e-02,
       0.00000000e+00, 7.68943993e-01, 0.00000000e+00, 0.00000000e+00,
       6.55121056e-01, 1.79371214e+00, 1.00987328e+00, 1.97365377e-01,
       2.87773387e-01, 1.60023487e-01, 0.00000000e+00, 3.70163728e-01,
       0.00000000e+00, 1.78336393e-01, 8.57519476e-02, 2.76593382e-02,
       7.75694623e-02, 1.67227590e+00, 7.90049670e-02, 9.12814205e-01,
       2.09116814e-03, 0.00000000e+00, 0.00000000e+00])

In [18]:
predictions = model.predict(test_pool)

comparison = pd.DataFrame({
    'actual': y_test.values,
    'predicted': predictions,
})
comparison['error'] = comparison['predicted'] - comparison['actual']
comparison['abs_error'] = comparison['error'].abs()
comparison['pct_error'] = (comparison['abs_error'] / comparison['actual']) * 100

comparison = comparison.sort_values('abs_error', ascending=False)
comparison.tail(60)

,actual,predicted,error,abs_error,pct_error
83,1150.0,1089.713553,-60.286447,60.286447,5.242300
41,1200.0,1255.141612,55.141612,55.141612,4.595134
81,1600.0,1654.408053,54.408053,54.408053,3.400503
145,719.0,770.973241,51.973241,51.973241,7.228545
84,1200.0,1148.322996,-51.677004,51.677004,4.306417
28,750.0,698.337024,-51.662976,51.662976,6.888397
16,970.0,918.648426,-51.351574,51.351574,5.293977
30,760.0,708.791821,-51.208179,51.208179,6.737918
128,1050.0,999.191018,-50.808982,50.808982,4.838951
51,1100.0,1053.896672,-46.103328,46.103328,4.191212


In [21]:
print('MAE:', comparison['abs_error'].mean())
print('RMSE:', (comparison['error']**2).mean()**0.5)
print('MAPE:', comparison['pct_error'].mean())
print('Max abs error:', comparison['abs_error'].max())
comparison.sort_values('abs_error', ascending=False).head(10)  # the actual worst cases

MAE: 95.4194262066109
RMSE: 150.1947930867974
MAPE: 14.773004780248684
Max abs error: 1086.3329597355846


,actual,predicted,error,abs_error,pct_error
91,250.0,1336.332960,1086.332960,1086.332960,434.533184
118,1800.0,1258.737535,-541.262465,541.262465,30.070137
7,100.0,608.841156,508.841156,508.841156,508.841156
3,900.0,1258.362313,358.362313,358.362313,39.818035
147,1950.0,1673.394273,-276.605727,276.605727,14.184909
125,1300.0,1030.110011,-269.889989,269.889989,20.760768
17,1400.0,1141.665104,-258.334896,258.334896,18.452493
138,1360.0,1102.914845,-257.085155,257.085155,18.903320
72,1450.0,1196.265922,-253.734078,253.734078,17.498902
23,1000.0,1249.477385,249.477385,249.477385,24.947739
